#Circuits Data ingestion 

In [0]:
%run ../00-common/01-env-variables

In [0]:
source = f"{landing_file_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StringType, DoubleType, StructField, StructType

circuits_schema = StructType([
    StructField('circuitId' , StringType()),
    StructField('url' , StringType()),
    StructField('circuitName' , StringType()),
    StructField('lat' , DoubleType()),
    StructField('lng' , DoubleType()),
    StructField('locality' , StringType()),
    StructField('country' , StringType()),
])

In [0]:
circuits_df = (
    spark.read.format('csv')
    .option('header', True)
   # .option('inferSchema', True)
    .schema(circuits_schema)
    .load(source)
)

In [0]:
display(circuits_df)

## Adding metadata

In [0]:
from pyspark.sql import functions as F
circuits_df = (circuits_df
.withColumn('ingestion_timestamp' , F.current_timestamp())
.withColumn('source_file' , F.col('_metadata.file_path'))
)

In [0]:
display(circuits_df)

In [0]:
(
    circuits_df.
    write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
select * from formula1.bronze.circuits
